<a href="https://colab.research.google.com/github/SimonHtet/Hotel/blob/main/exceltis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [513]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/Exceltis/Exeltis_Data_Engineer_Business_Case_Raw_Data.csv")
df.head()

,order_id,order_date,customer_name,client_segment,client_city,country,client_singup_date,product_name,quantity,unit_price,unit_cost,discount_pct,product_category,product_subcategory,supplier_name,product_active,sales_channel,sales_rep
0,O00001,9/13/2024 0:00,Mccoy Ltd,Hospitality,New Douglas,Portugal,44656,Nut Product 60,33,57.98,12.48,0.06,Snacks,Nuts,"Sims, Barber and Robinson",False,Sales Rep,Brenda Cook
1,O00002,1/27/2023 0:00,Porter and Sons,Distributor,Jorgefort,France,44372,Soft Drink Product 15,21,72.42,12.20,0.04,Beverages,Soft Drinks,"Rodriguez, Kelley and Myers",True,Phone,Kristin Burns
2,O00003,3/19/2023 0:00,Day Group,Distributor,Courtneystad,Portugal,45598,Nut Product 74,49,30.96,7.57,0.25,Snacks,Nuts,Greene-Bell,True,Sales Rep,Terri Mckay
3,O00004,9/6/2024 0:00,"Clark, Farrell and Sheppard",Distributor,New Molly,France,44921,Frozen Meal Product 32,90,58.39,38.22,0.30,Frozen,Frozen Meals,"Thomas, Harrison and Little",True,Online,Kevin Clark
4,O00005,1/15/2024 0:00,Hawkins and Sons,Distributor,Barnetthaven,France,45652,Cheese Product 73,6,27.35,35.41,0.15,Dairy,Cheese,Sharp-Cole,True,Online,Ethan Grant


In [514]:
import os
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if 'Exeltis' in file:
            print(os.path.join(root, file))

/content/drive/MyDrive/Exceltis/Exeltis_Data_Engineer_Business_Case_Raw_Data.csv


In [515]:
# == section 2: profile/explore ==
# markdown cell above this ##2.profiling -finding data quality issues

#structure: are the data types right?--
df.info()
df.head()

#-- whats missing?
print(df.isnull().sum())
print()
print(df['country'].value_counts())
print(df['client_segment'].value_counts())
print('Full duplicate rows:', df.duplicated().sum())
print('Duplicate order_id:', df['order_id'].duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2501 entries, 0 to 2500
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   order_id             2501 non-null   object 
 1   order_date           2501 non-null   object 
 2   customer_name        2501 non-null   object 
 3   client_segment       2501 non-null   object 
 4   client_city          2501 non-null   object 
 5   country              2501 non-null   object 
 6   client_singup_date   2501 non-null   object 
 7   product_name         2500 non-null   object 
 8   quantity             2501 non-null   int64  
 9   unit_price           2501 non-null   float64
 10  unit_cost            2500 non-null   float64
 11  discount_pct         2501 non-null   float64
 12  product_category     2500 non-null   object 
 13  product_subcategory  2500 non-null   object 
 14  supplier_name        2500 non-null   object 
 15  product_active       2500 non-null   o

In [516]:
df[['quantity','unit_price','unit_cost','discount_pct']].describe()

,quantity,unit_price,unit_cost,discount_pct
count,2501.000000,2501.000000,2500.000000,2501.000000
mean,50.004798,38.199592,19.320568,0.172367
std,28.827611,20.898183,11.705792,0.104649
min,-10.000000,2.010000,-5.000000,0.000000
25%,24.000000,19.900000,10.660000,0.080000
50%,50.000000,37.930000,17.770000,0.170000
75%,75.000000,55.880000,29.070000,0.260000
max,100.000000,74.990000,39.970000,1.500000


In [517]:
#some unit cost are negative
import numpy as np

df.loc[df['unit_cost'] <= 0, 'unit_cost'] = np.nan

In [518]:
print('Rows where cost == price :', (df['unit_cost'] == df['unit_cost']).sum())
print('Rows with inactive product sold :', (df['product_active'] == False).sum())



Rows where cost == price : 2467
Rows with inactive product sold : 508


In [519]:
print(df['sales_channel'].value_counts())
print(df[['order_date','client_singup_date']].head())

sales_channel
Online       845
Phone        843
Sales Rep    813
Name: count, dtype: int64
       order_date client_singup_date
0  9/13/2024 0:00              44656
1  1/27/2023 0:00              44372
2  3/19/2023 0:00              45598
3   9/6/2024 0:00              44921
4  1/15/2024 0:00              45652


In [520]:
def parse_mixed_dates(val):
    if pd.isna(val):
        return pd.NaT
    if isinstance(val, pd.Timestamp):
        return val
    s = str(val).strip()
    try:
        return pd.Timestamp('1899-12-30') + pd.Timedelta(days=int(float(s)))
    except (ValueError, TypeError):
        pass
    for fmt in ('%d/%m/%Y', '%m/%d/%Y', '%Y-%m-%d', '%Y/%m/%d', '%Y/%d/%m'):
        try:
            return pd.to_datetime(s, format=fmt)
        except (ValueError, TypeError):
            pass
    return pd.NaT

df['client_singup_date'] = df['client_singup_date'].apply(parse_mixed_dates)
print('nulls:', df['client_singup_date'].isna().sum())
print(df[['order_date','client_singup_date']].head())

nulls: 0
       order_date client_singup_date
0  9/13/2024 0:00         2022-04-05
1  1/27/2023 0:00         2021-06-25
2  3/19/2023 0:00         2024-11-02
3   9/6/2024 0:00         2022-12-26
4  1/15/2024 0:00         2024-12-26


In [521]:
print(df[df['customer_name'] == 'Richard, Simon and Madden'][['customer_name','client_singup_date']])

                  customer_name client_singup_date
68    Richard, Simon and Madden         2023-01-15
210   Richard, Simon and Madden         2023-01-15
593   Richard, Simon and Madden         2023-01-15
687   Richard, Simon and Madden         2023-01-15
951   Richard, Simon and Madden         2023-01-15
973   Richard, Simon and Madden         2023-01-15
1089  Richard, Simon and Madden         2023-01-15
1205  Richard, Simon and Madden         2023-01-15
1238  Richard, Simon and Madden         2023-01-15
1272  Richard, Simon and Madden         2023-01-15
1292  Richard, Simon and Madden         2023-01-15
1299  Richard, Simon and Madden         2023-01-15
1562  Richard, Simon and Madden         2023-01-15
1590  Richard, Simon and Madden         2023-01-15
1909  Richard, Simon and Madden         2023-01-15
2137  Richard, Simon and Madden         2023-01-15
2306  Richard, Simon and Madden         2023-01-15
2356  Richard, Simon and Madden         2023-01-15
2411  Richard, Simon and Madden

In [522]:
print(df['order_date'].iloc[80])

31-02-2024


In [523]:
df[df['order_date'] == '31-02-2024']

,order_id,order_date,customer_name,client_segment,client_city,country,client_singup_date,product_name,quantity,unit_price,unit_cost,discount_pct,product_category,product_subcategory,supplier_name,product_active,sales_channel,sales_rep
80,O00081,31-02-2024,Lee-Conway,Retail,Crystalshire,Italy,2023-02-16,Milk Product 4,72,25.94,20.96,0.21,Dairy,Milk,Mckee Group,True,Online,Amy Rodriguez


In [524]:
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', errors='coerce').dt.strftime('%Y/%m/%d')
print(df['order_date'].isna().sum())  # see how many became NaT

1


In [525]:
df[df.duplicated(keep=False)] #finding duplicated rows

,order_id,order_date,customer_name,client_segment,client_city,country,client_singup_date,product_name,quantity,unit_price,unit_cost,discount_pct,product_category,product_subcategory,supplier_name,product_active,sales_channel,sales_rep
100,O00101,2024/11/01,Landry-Yang,Distributor,West Stephanie,France,2025-02-17,Water Product 65,56,9.24,4.12,0.25,Beverages,Water,"Jefferson, Barnes and Hammond",True,Sales Rep,Michelle Eaton
2500,O00101,2024/11/01,Landry-Yang,Distributor,West Stephanie,France,2025-02-17,Water Product 65,56,9.24,4.12,0.25,Beverages,Water,"Jefferson, Barnes and Hammond",True,Sales Rep,Michelle Eaton


In [526]:
df = df.drop_duplicates() #drop duplicates

In [527]:
print('duplicate order_ids:', df['order_id'].duplicated().sum())
print('rows now:',len(df))


duplicate order_ids: 0
rows now: 2500


In [528]:
df[df['discount_pct']>1]

,order_id,order_date,customer_name,client_segment,client_city,country,client_singup_date,product_name,quantity,unit_price,unit_cost,discount_pct,product_category,product_subcategory,supplier_name,product_active,sales_channel,sales_rep
15,O00016,2024/07/17,Duran-Rodriguez,Distributor,Osbornland,France,2022-02-16,Bread Product 75,61,42.4,27.21,1.5,Bakery,Bread,Nguyen-Horne,True,Phone,Jesus Reyes


In [529]:
df.loc[df['discount_pct']>1, 'discount_pct'] = pd.NA

In [530]:
print('discounts > 1 now:', (df['discount_pct']>1).sum())

discounts > 1 now: 0


In [545]:
df = df.copy()
df['country'] = df['country'].replace('0', 'Unknown')
df['client_segment'] = df['client_segment'].replace('0', 'Unknown')
df['customer_name'] = df['customer_name'].replace('0', 'Unknown')

print(df['country'].value_counts())
print(df['client_segment'].value_counts())
print(df['customer_name'].value_counts())

country
France      729
Portugal    674
Italy       587
Spain       506
Unknown       1
Name: count, dtype: int64
client_segment
Distributor    826
Retail         561
Hospitality    560
Supermarket    532
Unknown         18
Name: count, dtype: int64
customer_name
Clark, Farrell and Sheppard    54
Johnson, Glass and Foster      32
Petersen Inc                   31
Jordan, Russell and Smith      30
Graham, Knapp and Rivera       29
                               ..
Pearson, White and Wells       12
Stokes and Sons                12
Massey-Moreno                   8
King and Sons                   8
Unknown                         1
Name: count, Length: 120, dtype: int64


In [532]:
print(df.shape)
print(df['order_date'].dtype)
print((df['country']=='0').sum())

(2500, 18)
object
0


In [533]:
import pandas as pd
import numpy as mp

print('row:', len(df))
print('dup order_id:', df['order_id'].duplicated().sum())
print('min quantity:', df['quantity'].min())
print('discount >1:', (df['discount_pct']>1).sum())
print("country '0':", (df['country'] == '0').sum(),"segment '0':",(df['client_segment']=='0').sum())

row: 2500
dup order_id: 0
min quantity: -10
discount >1: 0
country '0': 0 segment '0': 0


In [534]:
# some product categories is 0 or null
cat_lookup = (df.loc[df['product_category'] != '0']
              .drop_duplicates('product_subcategory')
              .set_index('product_subcategory')['product_category'])
m = df['product_category'] == '0'
df.loc[m, 'product_category'] = df.loc[m, 'product_subcategory'].map(cat_lookup)

#client city has 0 or null fix
df['client_city'] = df['client_city'].str.strip().replace('0', 'Unknown')

# see which subcategories have a valid category to borrow from
print(df.loc[df['product_category'] != '0', 'product_subcategory'].unique())

# see which '0' rows might not get filled
missing = df[df['product_category'] == '0']
print(missing['product_subcategory'].unique())

['Nuts' 'Soft Drinks' 'Frozen Meals' 'Cheese' 'Water' 'Yogurt' 'Pastries'
 'Milk' 'Bread' 'Chips' 'Crackers' 'Juices' nan 'Ice Cream']
[]


In [535]:
rejects_qty = df[df['quantity'] <= 0].copy()  # record of what i removed
df = df[df['quantity'] > 0]

# remove missing price / product_category
bad_mask = df['unit_price'].isna() | df['product_category'].isna() | (df['product_category'] == '0')
rejects_missing = df[bad_mask].copy()
df = df[~bad_mask]

# remove rows where order_date failed to parse (NaT from coerce)
rejects_date = df[df['order_date'].isna()].copy()
df = df[df['order_date'].notna()]

print('rows:', len(df))
print('quarantined (qty):', len(rejects_qty))
print('quarantined (missing):', len(rejects_missing))
print('quarantined (bad date):', len(rejects_date))

rows: 2497
quarantined (qty): 1
quarantined (missing): 1
quarantined (bad date): 1


In [536]:
rejects_qty = pd.read_csv("/content/drive/MyDrive/Exceltis/Exeltis_Data_Engineer_Business_Case_Raw_Data.csv")
rejects_qty = rejects_qty[rejects_qty['quantity'] <= 0]
print(rejects_qty)

   order_id      order_date customer_name client_segment client_city country  \
22   O00023  9/16/2023 0:00      Chen Inc    Distributor  Barnesport   Spain   

   client_singup_date     product_name  quantity  unit_price  unit_cost  \
22              45688  Milk Product 61       -10       26.87      12.92   

    discount_pct product_category product_subcategory           supplier_name  \
22          0.34            Dairy                Milk  Smith, Graham and Rowe   

   product_active sales_channel       sales_rep  
22           True         Phone  Lindsey Curtis  


In [537]:
# re-read raw and find dates that fail standard parsing
raw = pd.read_csv("/content/drive/MyDrive/Exceltis/Exeltis_Data_Engineer_Business_Case_Raw_Data.csv")
parsed = pd.to_datetime(raw['order_date'], format='mixed', errors='coerce')
print(raw.loc[parsed.isna(), 'order_date'].tolist())

['31-02-2024']


In [538]:
df['order_date'] = pd.to_datetime(df['order_date'], format='%Y/%m/%d')
print(df['order_date'].dtype)
print(df['order_date'].head())
print(df['client_singup_date'].dtype)
print(df['client_singup_date'].head())

datetime64[ns]
0   2024-09-13
1   2023-01-27
2   2023-03-19
3   2024-09-06
4   2024-01-15
Name: order_date, dtype: datetime64[ns]
datetime64[ns]
0   2022-04-05
1   2021-06-25
2   2024-11-02
3   2022-12-26
4   2024-12-26
Name: client_singup_date, dtype: datetime64[ns]


In [539]:
assert df['order_id'].is_unique, "order_id must be unique"
assert (df['order_date'].notna() & (df['order_date'] != '0')).all(), "no '0' or null in order_date"
assert (df['quantity']>0).all(), "quantity must be positive"
assert df['discount_pct'].dropna().between(0,1).all(), "discount must be 0-1"
assert (df['country'].notna() & (df['country'] != '0')).all(), "no '0' or null in country"
assert (df['client_city'].notna() & (df['client_city'] != '0')).all(), "no '0' or null in client_city"
assert (df['product_category'].notna() & (df['product_category'] != '0')).all(), "no '0' or null in product_category"
assert (df['client_segment'] != '0').all(), "no '0' placeholders in segments"
assert (df['unit_price'] > 0).all(), "unit price must be positive"
assert df['order_date'].dtype == 'datetime64[ns]', "order_date must be datetime"
assert df['client_singup_date'].dtype == 'datetime64[ns]', "client_singup_date must be datetime"

print(" All validation checks passed -", len(df), "clean rows")

 All validation checks passed - 2497 clean rows


In [540]:
print('df rows:', len(df))
print('unique customers:',df['customer_name'].nunique())
print('dim_customer rows:', len(dim_customer))
print('dim_customer rows:', len(dim_customer))
print('columns:', df.columns.tolist())

df rows: 2497
unique customers: 120
dim_customer rows: 120
dim_customer rows: 120
columns: ['order_id', 'order_date', 'customer_name', 'client_segment', 'client_city', 'country', 'client_singup_date', 'product_name', 'quantity', 'unit_price', 'unit_cost', 'discount_pct', 'product_category', 'product_subcategory', 'supplier_name', 'product_active', 'sales_channel', 'sales_rep']


In [541]:
# dim customer - one row per unique customer
cust_cols = ['customer_name', 'client_segment', 'client_city', 'country', 'client_singup_date']
dim_customer = (
    df[cust_cols]
    .drop_duplicates(subset=['customer_name'])  # one row per customer
    .reset_index(drop=True)
)

dim_customer['client_singup_date'] = dim_customer['client_singup_date'].dt.normalize()
dim_customer['customer_key'] = dim_customer.index + 1

print('dim_customer rows:', len(dim_customer))
dim_customer.head()

dim_customer rows: 120


,customer_name,client_segment,client_city,country,client_singup_date,customer_key
0,Mccoy Ltd,Hospitality,New Douglas,Portugal,2022-04-05,1
1,Porter and Sons,Distributor,Jorgefort,France,2021-06-25,2
2,Day Group,Distributor,Courtneystad,Portugal,2024-11-02,3
3,"Clark, Farrell and Sheppard",Distributor,New Molly,France,2022-12-26,4
4,Hawkins and Sons,Distributor,Barnetthaven,France,2024-12-26,5


In [542]:
# ==== dim_product
prod_cols = ['product_name','product_category', 'product_subcategory','supplier_name','product_active','unit_cost']
dim_product = (
    df[prod_cols]
    .drop_duplicates(subset=['product_name'])  # one row per product
    .reset_index(drop=True)
)

dim_product['product_key'] = dim_product.index + 1
print('dim_product rows:',len(dim_product))

# ==== dim_channel
dim_channel = (
    df[['sales_channel']]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_channel['channel_key'] = dim_channel.index + 1
print('dim_channel rows:',len(dim_channel))
dim_product.head()

#dim_date
dim_date = (
    df[['order_date']]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_date['date_key'] = dim_date.index + 1
dim_date['year'] = dim_date['order_date'].dt.year
dim_date['quarter'] = dim_date['order_date'].dt.quarter
dim_date['month'] = dim_date['order_date'].dt.month
dim_date['month_name'] = dim_date['order_date'].dt.month_name()
dim_date['day_name'] = dim_date['order_date'].dt.day_name()
print('dim_date rows:',len(dim_date))
dim_date.head()


dim_product rows: 79
dim_channel rows: 3
dim_date rows: 709


,order_date,date_key,year,quarter,month,month_name,day_name
0,2024-09-13,1,2024,3,9,September,Friday
1,2023-01-27,2,2023,1,1,January,Friday
2,2023-03-19,3,2023,1,3,March,Sunday
3,2024-09-06,4,2024,3,9,September,Friday
4,2024-01-15,5,2024,1,1,January,Monday


In [543]:
# Sales the transaction
fact_sales = df.copy()

# join each table to bring in keys
fact_sales = fact_sales.merge(dim_customer[['customer_name', 'customer_key']], on='customer_name', how='left')
fact_sales = fact_sales.merge(dim_product[['product_name', 'product_key']], on='product_name', how='left')
fact_sales = fact_sales.merge(dim_channel[['sales_channel', 'channel_key']], on='sales_channel', how='left')
fact_sales = fact_sales.merge(dim_date[['order_date', 'date_key']], on='order_date', how='left')

#compute the measures
fact_sales['revenue'] = (fact_sales['quantity'] * fact_sales['unit_price'] * (1 - fact_sales['discount_pct'].fillna(0))).round(2)
fact_sales['cost'] = (fact_sales['quantity'] * fact_sales['unit_cost']).round(2)
fact_sales['profit'] = (fact_sales['revenue'] - fact_sales['cost']).round(2)

#order_id + keys+transactions measures
fact_cols = ['order_id', 'order_date', 'date_key', 'customer_key', 'product_key', 'channel_key', 'quantity', 'unit_price', 'unit_cost', 'discount_pct', 'revenue', 'cost', 'profit']
fact_sales = fact_sales[fact_cols]

print('fact_sales rows:',len(fact_sales))
fact_sales

fact_sales rows: 2497


,order_id,order_date,date_key,customer_key,product_key,channel_key,quantity,unit_price,unit_cost,discount_pct,revenue,cost,profit
0,O00001,2024-09-13,1,1,1,1,33,57.98,12.48,0.06,1798.54,411.84,1386.70
1,O00002,2023-01-27,2,2,2,2,21,72.42,12.20,0.04,1459.99,256.20,1203.79
2,O00003,2023-03-19,3,3,3,1,49,30.96,7.57,0.25,1137.78,370.93,766.85
3,O00004,2024-09-06,4,4,4,3,90,58.39,38.22,0.30,3678.57,3439.80,238.77
4,O00005,2024-01-15,5,5,5,3,6,27.35,35.41,0.15,139.49,212.46,-72.97
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2492,O02496,2024-06-01,594,84,4,2,20,69.60,38.22,0.24,1057.92,764.40,293.52
2493,O02497,2023-03-25,216,66,38,3,52,32.61,8.88,0.12,1492.23,461.76,1030.47
2494,O02498,2023-11-18,558,118,24,3,62,71.24,9.69,0.23,3401.00,600.78,2800.22
2495,O02499,2023-02-01,161,71,7,2,56,13.25,39.97,0.17,615.86,2238.32,-1622.46


In [544]:
#export to csv
fact_sales.to_csv('tbl_sales.csv', index=False)
dim_customer.to_csv('tbl_customer.csv', index=False)
dim_product.to_csv('tbl_product.csv', index=False)
dim_channel.to_csv('tbl_channel.csv', index=False)
dim_date.to_csv('tbl_date.csv', index=False)

# quarantined data into csv (re-read raw to capture original derived values)
rejects = pd.read_csv("/content/drive/MyDrive/Exceltis/Exeltis_Data_Engineer_Business_Case_Raw_Data.csv")
rejects = rejects[
    (rejects['quantity'] <= 0)
    | rejects['unit_price'].isna()
    | rejects['unit_cost'].isna()
]
print(len(rejects), "quarantined row(s)")
rejects.to_csv('tbl_rejects_quarantined.csv', index=False)

print('all exported')


2 quarantined row(s)
all exported
